# Data Merging and Preprocessing (12-Class Flat Classification)
This notebook merges the raw datasets, standardizes a single `label` column (FR + 11 sub_NFRs), strictly caps SE to 200 globally, caps FR to 200 specifically in the train set, and outputs train/val/test sets.

In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

data_dir = '../data/raw'
processed_dir = '../data/processed'

# 1. PROMISE-relabeled-NICE
df_nice = pd.read_csv(os.path.join(data_dir, 'PROMISE-relabeled-NICE.csv'))
nfr_cols = ['Availability (A)', 'Fault Tolerance (FT)', 'Legal (L)', 'Look & Feel (LF)', 
            'Maintainability (MN)', 'Operability (O)', 'Performance (PE)', 'Portability (PO)', 
            'Scalability (SC)', 'Security (SE)', 'Usability (US)', 'Other (OT)']

def get_nice_sub_nfr(row):
    for col in nfr_cols:
        if row[col] == 1:
            if '(' in col and ')' in col:
                return col.split('(')[1].split(')')[0]
            return col
    return None

df_nice['text'] = df_nice['RequirementText']
df_nice['label_binary'] = df_nice.apply(lambda r: 'FR' if r['IsFunctional'] == 1 and r['IsQuality'] == 0 else 'NFR', axis=1)
df_nice['sub_NFR'] = df_nice.apply(get_nice_sub_nfr, axis=1)
df_nice['source'] = 'NICE'
nice_final = df_nice[['text', 'label_binary', 'sub_NFR', 'source']].copy()

# 2. promise_exp
df_exp = pd.read_csv(os.path.join(data_dir, 'promise_exp.csv'))
df_exp['text'] = df_exp['RequirementText']
df_exp['label_binary'] = df_exp['class'].apply(lambda x: 'FR' if str(x).strip().upper() == 'F' else 'NFR')
df_exp['sub_NFR'] = df_exp['class']
df_exp['source'] = 'promise_exp'
exp_final = df_exp[['text', 'label_binary', 'sub_NFR', 'source']].copy()

# 3. dcai24
df_dcai = pd.read_excel(os.path.join(data_dir, 'dcai24_src_dataset.xlsx'))
df_dcai['text'] = df_dcai['Requirement']
df_dcai['sub_NFR'] = df_dcai['Specific_Type']

def get_dcai_binary(row):
    t = str(row['Type']).strip().upper()
    s = str(row['Specific_Type']).strip().upper()
    if t == 'FR' or s == 'NONSE':
        return 'FR'
    return 'NFR'

df_dcai['label_binary'] = df_dcai.apply(get_dcai_binary, axis=1)
df_dcai['sub_NFR'] = df_dcai['sub_NFR'].replace({'1': 'L', 'l': 'L', 'NONSEPO': 'PO', 'NONSE': 'F'})
df_dcai['source'] = 'dcai24'
dcai_final = df_dcai[['text', 'label_binary', 'sub_NFR', 'source']].copy()

# Combine all WITHOUT EARS
merged = pd.concat([nice_final, exp_final, dcai_final], ignore_index=True)

# Deduplication
merged = merged.drop_duplicates(subset=['text'], keep='first')

# Check noise
nan_text = merged['text'].isna() | (merged['text'].astype(str).str.strip() == '')
nfr_missing_sub = (merged['label_binary'] == 'NFR') & (merged['sub_NFR'].isna() | (merged['sub_NFR'].astype(str).str.strip() == ''))

# Delete all noise
merged_clean = merged[~(nan_text | nfr_missing_sub)].copy()

# Fix FR missing sub_NFR
merged_clean.loc[(merged_clean['label_binary'] == 'FR') & (merged_clean['sub_NFR'].isna()), 'sub_NFR'] = 'F'

# --- NEW FLAT 12-CLASS LABELING ---
merged_clean['label'] = merged_clean.apply(lambda r: 'FR' if r['label_binary'] == 'FR' else r['sub_NFR'], axis=1)

print(f"Total rows before splitting: {len(merged_clean)}")

# --- Train/Val/Test Split Stratified on the 12 Classes ---
X = merged_clean[['text', 'label', 'source']]
y = merged_clean['label']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

val_ratio = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_ratio, stratify=y_train_val, random_state=42
)

train_df = X_train.copy()
val_df = X_val.copy()
test_df = X_test.copy()

# --- Downsample FR and SE in Train Set to 200 ---
def cap_class(df, class_name, cap_size=200):
    cls_df = df[df['label'] == class_name]
    other_df = df[df['label'] != class_name]
    if len(cls_df) > cap_size:
        cls_df = cls_df.sample(n=cap_size, random_state=42)
    return pd.concat([other_df, cls_df], ignore_index=True)

train_df = cap_class(train_df, 'FR', 200)
train_df = cap_class(train_df, 'SE', 200)
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True) # shuffle

print(f"Train size after capping FR and SE to 200: {len(train_df)}")

train_df.to_csv(os.path.join(processed_dir, 'train.csv'), index=False)
val_df.to_csv(os.path.join(processed_dir, 'val.csv'), index=False)
test_df.to_csv(os.path.join(processed_dir, 'test.csv'), index=False)
print("Saved updated 12-class train/val/test splits (with real SE).")


Total rows before splitting: 3517
Train size after capping FR and SE to 200: 1018
Saved updated 12-class train/val/test splits (with real SE).
